# Analyzing DHS microdata for Nigeria

Nigeria 2018 - J:/DATA/DHS_PROG_DHS/NGA/2018/

## Documentation

DHS 7 recode manual for variable definitions: https://www.dhsprogram.com/pubs/pdf/DHSG4/Recode7_DHS_10Sep2018_DHSG4.pdf.
However, it does not state which variables are in which file.
The `.MAP` files alongside each data file list the variables in it and what they mean.

Hemoglobin variables of interest: 
- HA53: Hemoglobin level in g/dl with 1 implied dcimal
- HA54: Currently pregnant
- HA55 Result of Hemoglobin measuring.
- HA56 Hemoglobin level adjusted by altitude in g/dl with 1 implied decimal. 

Wealth index variables of interest: 
- HV270: The wealth index is a composite measure of a household's cumulative living standard.
The wealth index is calculated using easy-to-collect data on a household’s ownership of
selected assets, such as televisions and bicycles; materials used for housing construction; and
types of water access and sanitation facilities.
Generated with a statistical procedure known as principal components analysis, the wealth
index places individual households on a continuous scale of relative wealth. DHS separates
all interviewed households into five wealth quintiles to compare the influence of wealth on
various population, health and nutrition indicators. The wealth index is presented in the DHS
Final Reports and survey datasets as a background characteristic
- HV271: Wealth index factor score (5 decimals) 

Pregnancy variables of interest: 
- HML18: Pregnancy status from individual questionnaire. For complete woman’s interviews this is
taken from V213. For incomplete woman's interview with anemia testing the pregnancy
status is taken from this section.
BASE: Women with a completed individual questionnaire or when available information
from the anemia testing section.

List of datasets: https://www.dhsprogram.com/data/dataset/Nigeria_Standard-DHS_2018.cfm?flag=1

Instructions on how to calculate everything can be found at: https://www.dhsprogram.com/pubs/pdf/DHSG1/Guide_to_DHS_Statistics_DHS-7_v2.pdf

In [1]:
import pandas as pd, numpy as np

%load_ext autoreload
%autoreload 2

!date

Thu 25 Jul 2024 05:43:15 PM PDT


## Load data, name columns

In [2]:
directory = '/snfs1/DATA/DHS_PROG_DHS/NGA/2018/'

### WRA

In [3]:
%%time

raw_wra_data = pd.read_stata(directory + 'NGA_DHS7_2018_WN_NGIR7AFL_Y2019M11D05.DTA')

CPU times: user 9.62 s, sys: 2.19 s, total: 11.8 s
Wall time: 11.8 s


In [4]:
wra_data =  raw_wra_data.copy()
wra_data

,caseid,v000,v001,v002,v003,v004,v005,v006,v007,v008,...,s434k_3,s434k_4,s434k_5,s434k_6,s434l_1,s434l_2,s434l_3,s434l_4,s434l_5,s434l_6
0,1 1 2,NG7,1,1,2,1,1335530,9,2018,1425,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1 6 4,NG7,1,6,4,1,1335530,9,2018,1425,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,1 11 1,NG7,1,11,1,1,1335530,9,2018,1425,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,1 25 2,NG7,1,25,2,1,1335530,9,2018,1425,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,1 30 1,NG7,1,30,1,1,1335530,9,2018,1425,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
41816,1400 35 3,NG7,1400,35,3,1400,768129,10,2018,1426,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
41817,1400 35 7,NG7,1400,35,7,1400,768129,10,2018,1426,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
41818,1400 38 2,NG7,1400,38,2,1400,768129,10,2018,1426,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
41819,1400 40 1,NG7,1400,40,1,1400,768129,10,2018,1426,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [5]:
wra_columns = {
    "v001": "cluster_number",
    "v002": "household_number",
    "v003": "line_number",
    "v005": "weight",
    "v008": "interview_date",
    "v011": "date_of_birth",
    "v190": "wealth_quintile",
}
wra_data = wra_data[wra_columns.keys()].rename(columns=wra_columns)

In [6]:
def recode_wealth_quintile(df):
    return df.map({
        "poorest": "lowest",
        "poorer": "second",
        "middle": "middle",
        "richer": "fourth",
        "richest": "highest",
    })

In [7]:
wra_data["wealth_quintile"] = recode_wealth_quintile(wra_data.wealth_quintile)

In [8]:
wra_data["weight"] = wra_data.weight / 1_000_000

### Births

In [9]:
birth_data =  pd.read_stata(directory + 'NGA_DHS7_2018_BR_NGBR7AFL_Y2019M11D05.DTA')
birth_data

,caseid,bidx,v000,v001,v002,v003,v004,v005,v006,v007,...,s434ib,s434ic,s434id,s434ie,s434if,s434ig,s434ix,s434iz,s434k,s434l
0,1 1 2,1,NG7,1,1,2,1,1335530,9,2018,...,yes,no,no,no,no,no,no,no,no,NaN
1,1 1 2,2,NG7,1,1,2,1,1335530,9,2018,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,1 1 2,3,NG7,1,1,2,1,1335530,9,2018,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,1 11 1,1,NG7,1,11,1,1,1335530,9,2018,...,yes,no,no,no,no,no,no,no,no,NaN
4,1 11 1,2,NG7,1,11,1,1,1335530,9,2018,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
127540,1400 38 2,1,NG7,1400,38,2,1400,768129,10,2018,...,yes,no,no,no,no,no,no,no,no,NaN
127541,1400 45 2,1,NG7,1400,45,2,1400,768129,10,2018,...,yes,no,no,no,no,no,no,no,no,NaN
127542,1400 45 2,2,NG7,1400,45,2,1400,768129,10,2018,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
127543,1400 45 2,3,NG7,1400,45,2,1400,768129,10,2018,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [10]:
birth_columns = {
    "v005": "weight",
    "v008": "interview_date",
    "v190": "wealth_quintile",
    "b3": "birth_date",
}
birth_data = birth_data[birth_columns.keys()].rename(columns=birth_columns)
birth_data["wealth_quintile"] = recode_wealth_quintile(birth_data.wealth_quintile)
birth_data["weight"] = birth_data.weight / 1_000_000
birth_data

,weight,interview_date,wealth_quintile,birth_date
0,1.335530,1425,highest,1412
1,1.335530,1425,highest,1292
2,1.335530,1425,highest,1206
3,1.335530,1425,highest,1416
4,1.335530,1425,highest,1361
...,...,...,...,...
127540,0.768129,1426,fourth,1418
127541,0.768129,1426,highest,1384
127542,0.768129,1426,highest,1366
127543,0.768129,1426,highest,1355


### Household members

In [11]:
%%time

hhm_data =  pd.read_stata(directory + 'NGA_DHS7_2018_HHM_NGPR7AFL_Y2019M11D05.DTA')
hhm_data

CPU times: user 1.81 s, sys: 51.3 ms, total: 1.87 s
Wall time: 1.87 s


,hhid,hvidx,hv000,hv001,hv002,hv003,hv004,hv005,hv006,hv007,...,idxdis,hdis1,hdis2,hdis3,hdis4,hdis5,hdis6,hdis7,hdis8,hdis9
0,1 1,1,NG7,1,1,1,1,1368354,9,2018,...,1,yes,some difficulty,no,no difficulty hearing,no difficulty communicating,some difficulty,some difficulty,some difficulty,some difficulty
1,1 1,2,NG7,1,1,1,1,1368354,9,2018,...,2,no,no difficulty seeing,no,no difficulty hearing,no difficulty communicating,no difficulty remembering/concentrating,no difficulty walking or climbing,no difficulty washing or dressing,no difficulty
2,1 1,3,NG7,1,1,1,1,1368354,9,2018,...,3,no,no difficulty seeing,no,no difficulty hearing,no difficulty communicating,no difficulty remembering/concentrating,no difficulty walking or climbing,no difficulty washing or dressing,no difficulty
3,1 1,4,NG7,1,1,1,1,1368354,9,2018,...,4,no,no difficulty seeing,no,no difficulty hearing,no difficulty communicating,no difficulty remembering/concentrating,no difficulty walking or climbing,no difficulty washing or dressing,no difficulty
4,1 1,5,NG7,1,1,1,1,1368354,9,2018,...,5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
188005,1400 45,2,NG7,1400,45,1,1400,787007,10,2018,...,2,no,no difficulty seeing,no,no difficulty hearing,no difficulty communicating,no difficulty remembering/concentrating,no difficulty walking or climbing,no difficulty washing or dressing,no difficulty
188006,1400 45,3,NG7,1400,45,1,1400,787007,10,2018,...,3,no,no difficulty seeing,no,no difficulty hearing,no difficulty communicating,no difficulty remembering/concentrating,no difficulty walking or climbing,no difficulty washing or dressing,no difficulty
188007,1400 46,1,NG7,1400,46,1,1400,787007,10,2018,...,1,no,no difficulty seeing,no,no difficulty hearing,no difficulty communicating,no difficulty remembering/concentrating,no difficulty walking or climbing,no difficulty washing or dressing,no difficulty
188008,1400 46,2,NG7,1400,46,1,1400,787007,10,2018,...,2,no,no difficulty seeing,no,no difficulty hearing,no difficulty communicating,no difficulty remembering/concentrating,no difficulty walking or climbing,no difficulty washing or dressing,no difficulty


In [12]:
hhm_columns = {
    "hv001": "cluster_number",
    "hv002": "household_number",
    "hvidx": "line_number",
    "hml18": "currently_pregnant",
    "ha1": "age",
    "hv270": "wealth_quintile",
    "ha53": "hemoglobin_raw",
    "ha56": "hemoglobin_adjusted",
    "ha57": "anemia",
}
hhm_data = hhm_data[hhm_columns.keys()].rename(columns=hhm_columns)

In [13]:
hhm_data["wealth_quintile"] = recode_wealth_quintile(hhm_data.wealth_quintile)

In [14]:
for col in ["hemoglobin_raw", "hemoglobin_adjusted"]:
    hhm_data[col] = hhm_data[col].astype(str).replace({
        'not present': np.nan,
        'refused': np.nan,
        'other': np.nan
    }).astype(float)

### Siblings

Sibling survival data appears to only be available as a kind of side table-within-a-table on WRA.

It is labeled "MM" because it is used to calculate maternal mortality (among other things).

In [15]:
respondent_column_names = {
    "v008": "interview_date",
    "v190": "wealth_quintile",
    "v005": "weight",
}

# These are suffixed with an underscore and an integer, e.g. mm1_01
sibling_column_names = {
    # MM1                    Sex of sibling                                  7156    1    N    I   20    0   No   No
    "mm1": "sex",
    # MM2                    Survival status of sibling                      7176    1    N    I   20    0   No   No
    "mm2": "survival_status",
    # MM3                    Sibling's current age                           7196    2    N    I   20    0   No   No
    "mm3": "current_age",
    # MM4                    Sibling's date of birth (CMC)                   7236    4    N    I   20    0   No   No
    "mm4": "date_of_birth",
    # MM8                    Date of death of sibling (CMC)                  7416    4    N    I   20    0   No   No
    "mm8": "date_of_death",
    # MM7                    Sibling's age at death                          7376    2    N    I   20    0   No   No
    "mm7": "age_at_death",
    # MM9                    Sibling's death and pregnancy                   7496    2    N    I   20    0   No   No
    "mm9": "pregnancy_category",
    # MM16                   Sibling's death due to violence or accident     7816    1    N    I   20    0   No   No
    "mm16": "death_violence_or_accident",
}

In [16]:
sibling_data = raw_wra_data[
    [c for c in raw_wra_data.columns if c in respondent_column_names.keys() or c.split('_')[0] in sibling_column_names.keys()]
].copy()
sibling_data

,v005,v008,v190,mm1_01,mm1_02,mm1_03,mm1_04,mm1_05,mm1_06,mm1_07,...,mm16_11,mm16_12,mm16_13,mm16_14,mm16_15,mm16_16,mm16_17,mm16_18,mm16_19,mm16_20
0,1335530,1425,richest,female,female,male,female,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1335530,1425,richest,male,male,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,1335530,1425,richest,female,female,female,male,female,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,1335530,1425,richest,female,male,male,female,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,1335530,1425,richest,male,female,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
41816,768129,1426,richer,female,female,male,female,male,male,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
41817,768129,1426,richer,female,female,male,male,male,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
41818,768129,1426,richer,male,male,male,male,male,male,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
41819,768129,1426,richest,female,female,male,male,male,male,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [17]:
# inspired by https://stackoverflow.com/a/67393747/
sibling_data_reshaped = sibling_data[[c for c in sibling_data.columns if c.split('_')[0] in sibling_column_names.keys()]].copy()
sibling_data_reshaped.columns = sibling_data_reshaped.columns.str.split("_", expand = True)
sibling_data_reshaped

mm1                                                            ...  \
           01      02      03      04      05    06   07   08   09   10  ...   
0      female  female    male  female     NaN   NaN  NaN  NaN  NaN  NaN  ...   
1        male    male     NaN     NaN     NaN   NaN  NaN  NaN  NaN  NaN  ...   
2      female  female  female    male  female   NaN  NaN  NaN  NaN  NaN  ...   
3      female    male    male  female     NaN   NaN  NaN  NaN  NaN  NaN  ...   
4        male  female     NaN     NaN     NaN   NaN  NaN  NaN  NaN  NaN  ...   
...       ...     ...     ...     ...     ...   ...  ...  ...  ...  ...  ...   
41816  female  female    male  female    male  male  NaN  NaN  NaN  NaN  ...   
41817  female  female    male    male    male   NaN  NaN  NaN  NaN  NaN  ...   
41818    male    male    male    male    male  male  NaN  NaN  NaN  NaN  ...   
41819  female  female    male    male    male  male  NaN  NaN  NaN  NaN  ...   
41820    male    male  female    male  female   NaN  NaN  NaN  NaN  NaN  ...   

      mm16                                               
        11   12   13   14   15   16   17   18   19   20  
0      NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  
1      NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  
2      NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  
3      NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  
4      NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  
...    ...  ...  ...  ...  ...  ...  ...  ...  ...  ...  
41816  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  
41817  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  
41818  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  
41819  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  
41820  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  

[41821 rows x 160 columns]

In [18]:
sibling_data_reshaped[list(respondent_column_names.keys())] = sibling_data[list(respondent_column_names.keys())]
sibling_data_reshaped

mm1                                                            ...  \
           01      02      03      04      05    06   07   08   09   10  ...   
0      female  female    male  female     NaN   NaN  NaN  NaN  NaN  NaN  ...   
1        male    male     NaN     NaN     NaN   NaN  NaN  NaN  NaN  NaN  ...   
2      female  female  female    male  female   NaN  NaN  NaN  NaN  NaN  ...   
3      female    male    male  female     NaN   NaN  NaN  NaN  NaN  NaN  ...   
4        male  female     NaN     NaN     NaN   NaN  NaN  NaN  NaN  NaN  ...   
...       ...     ...     ...     ...     ...   ...  ...  ...  ...  ...  ...   
41816  female  female    male  female    male  male  NaN  NaN  NaN  NaN  ...   
41817  female  female    male    male    male   NaN  NaN  NaN  NaN  NaN  ...   
41818    male    male    male    male    male  male  NaN  NaN  NaN  NaN  ...   
41819  female  female    male    male    male  male  NaN  NaN  NaN  NaN  ...   
41820    male    male  female    male  female   NaN  NaN  NaN  NaN  NaN  ...   

      mm16                                v008     v190     v005  
        14   15   16   17   18   19   20                          
0      NaN  NaN  NaN  NaN  NaN  NaN  NaN  1425  richest  1335530  
1      NaN  NaN  NaN  NaN  NaN  NaN  NaN  1425  richest  1335530  
2      NaN  NaN  NaN  NaN  NaN  NaN  NaN  1425  richest  1335530  
3      NaN  NaN  NaN  NaN  NaN  NaN  NaN  1425  richest  1335530  
4      NaN  NaN  NaN  NaN  NaN  NaN  NaN  1425  richest  1335530  
...    ...  ...  ...  ...  ...  ...  ...   ...      ...      ...  
41816  NaN  NaN  NaN  NaN  NaN  NaN  NaN  1426   richer   768129  
41817  NaN  NaN  NaN  NaN  NaN  NaN  NaN  1426   richer   768129  
41818  NaN  NaN  NaN  NaN  NaN  NaN  NaN  1426   richer   768129  
41819  NaN  NaN  NaN  NaN  NaN  NaN  NaN  1426  richest   768129  
41820  NaN  NaN  NaN  NaN  NaN  NaN  NaN  1426  richest   768129  

[41821 rows x 163 columns]

In [19]:
# Get a row per sibling
sibling_data_reshaped = sibling_data_reshaped.set_index(list(respondent_column_names.keys())).swaplevel(axis=1).stack(0).reset_index().drop(columns=[f"level_{len(respondent_column_names)}"])
sibling_data_reshaped

/tmp/ipykernel_1229705/1476990635.py:2: FutureWarning: The previous implementation of stack is deprecated and will be removed in a future version of pandas. See the What's New notes for pandas 2.1.0 for details. Specify future_stack=True to adopt the new implementation and silence this warning.
  sibling_data_reshaped = sibling_data_reshaped.set_index(list(respondent_column_names.keys())).swaplevel(axis=1).stack(0).reset_index().drop(columns=[f"level_{len(respondent_column_names)}"])


,v008,v190,v005,mm1,mm2,mm3,mm4,mm7,mm8,mm9,mm16
0,1425,richest,1335530,female,alive,53.0,783.0,NaN,NaN,NaN,NaN
1,1425,richest,1335530,female,alive,50.0,819.0,NaN,NaN,NaN,NaN
2,1425,richest,1335530,male,alive,46.0,867.0,NaN,NaN,NaN,NaN
3,1425,richest,1335530,female,alive,43.0,903.0,NaN,NaN,NaN,NaN
4,1425,richest,1335530,male,alive,24.0,1131.0,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...
219556,1426,richest,768129,male,alive,38.0,964.0,NaN,NaN,NaN,NaN
219557,1426,richest,768129,male,alive,36.0,988.0,NaN,NaN,NaN,NaN
219558,1426,richest,768129,female,alive,34.0,1012.0,NaN,NaN,NaN,NaN
219559,1426,richest,768129,male,alive,28.0,1084.0,NaN,NaN,NaN,NaN


In [20]:
sibling_data = (
    sibling_data_reshaped[list(respondent_column_names.keys()) + list(sibling_column_names.keys())]
        .rename(columns=respondent_column_names)
        .rename(columns=sibling_column_names)
)
sibling_data["wealth_quintile"] = recode_wealth_quintile(sibling_data.wealth_quintile)
sibling_data["weight"] = sibling_data.weight / 1_000_000
sibling_data

,interview_date,wealth_quintile,weight,sex,survival_status,current_age,date_of_birth,date_of_death,age_at_death,pregnancy_category,death_violence_or_accident
0,1425,highest,1.335530,female,alive,53.0,783.0,NaN,NaN,NaN,NaN
1,1425,highest,1.335530,female,alive,50.0,819.0,NaN,NaN,NaN,NaN
2,1425,highest,1.335530,male,alive,46.0,867.0,NaN,NaN,NaN,NaN
3,1425,highest,1.335530,female,alive,43.0,903.0,NaN,NaN,NaN,NaN
4,1425,highest,1.335530,male,alive,24.0,1131.0,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...
219556,1426,highest,0.768129,male,alive,38.0,964.0,NaN,NaN,NaN,NaN
219557,1426,highest,0.768129,male,alive,36.0,988.0,NaN,NaN,NaN,NaN
219558,1426,highest,0.768129,female,alive,34.0,1012.0,NaN,NaN,NaN,NaN
219559,1426,highest,0.768129,male,alive,28.0,1084.0,NaN,NaN,NaN,NaN


In [21]:
# "A total of 219,561 siblings were recorded..." (p. 372)
len(sibling_data)

219561

## WRA

### Hemoglobin among pregnancies

In [22]:
id_columns = ["cluster_number", "household_number", "line_number"]
other_overlapping_columns = (set(wra_data.columns) & set(hhm_data.columns)) - set(id_columns)
other_overlapping_columns

{'wealth_quintile'}

In [23]:
wra_hhm_joined = wra_data.merge(
    hhm_data,
    on=id_columns,
    suffixes=("_wra", "_hhm"),
    how="left",
)
wra_hhm_joined

,cluster_number,household_number,line_number,weight,interview_date,date_of_birth,wealth_quintile_wra,currently_pregnant,age,wealth_quintile_hhm,hemoglobin_raw,hemoglobin_adjusted,anemia
0,1,1,2,1.335530,1425,939,highest,"not pregnant, don't know",NaN,highest,NaN,NaN,NaN
1,1,6,4,1.335530,1425,1230,highest,"not pregnant, don't know",16.0,highest,127.0,127.0,not anemic
2,1,11,1,1.335530,1425,977,highest,"not pregnant, don't know",NaN,highest,NaN,NaN,NaN
3,1,25,2,1.335530,1425,1091,highest,"not pregnant, don't know",NaN,highest,NaN,NaN,NaN
4,1,30,1,1.335530,1425,1073,highest,"not pregnant, don't know",NaN,highest,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...
41816,1400,35,3,0.768129,1426,1203,fourth,"not pregnant, don't know",NaN,fourth,NaN,NaN,NaN
41817,1400,35,7,0.768129,1426,1165,fourth,"not pregnant, don't know",NaN,fourth,NaN,NaN,NaN
41818,1400,38,2,0.768129,1426,1164,fourth,"not pregnant, don't know",21.0,fourth,123.0,123.0,not anemic
41819,1400,40,1,0.768129,1426,1119,highest,"not pregnant, don't know",NaN,highest,NaN,NaN,NaN


In [24]:
for col in other_overlapping_columns:
    assert (wra_hhm_joined[f'{col}_wra'] == wra_hhm_joined[f'{col}_hhm']).all()
    wra_hhm_joined[col] = wra_hhm_joined[f'{col}_wra']
    wra_hhm_joined = wra_hhm_joined.drop(columns=[f'{col}_wra', f'{col}_hhm'])

In [25]:
wra_hhm_joined.currently_pregnant.value_counts(dropna=False)

currently_pregnant
not pregnant, don't know    37630
pregnant                     4191
Name: count, dtype: int64

In [26]:
pregnant_data = wra_hhm_joined[wra_hhm_joined.currently_pregnant == 'pregnant'].copy()
pregnant_data

,cluster_number,household_number,line_number,weight,interview_date,date_of_birth,currently_pregnant,age,hemoglobin_raw,hemoglobin_adjusted,anemia,wealth_quintile
5,1,35,2,1.335530,1425,1094,pregnant,27.0,95.0,95.0,moderate,highest
36,2,117,2,2.726707,1425,1095,pregnant,27.0,109.0,109.0,mild,fourth
38,2,147,2,2.726707,1425,1125,pregnant,25.0,101.0,101.0,mild,fourth
39,2,156,2,2.726707,1425,1097,pregnant,NaN,NaN,NaN,NaN,highest
44,2,205,2,2.726707,1425,1047,pregnant,31.0,102.0,102.0,mild,highest
...,...,...,...,...,...,...,...,...,...,...,...,...
41744,1397,9,2,0.678629,1428,998,pregnant,NaN,NaN,NaN,NaN,fourth
41767,1398,31,1,1.657897,1428,1015,pregnant,34.0,126.0,126.0,not anemic,fourth
41786,1399,4,2,1.023897,1428,1205,pregnant,NaN,NaN,NaN,NaN,second
41791,1399,21,2,1.023897,1428,1054,pregnant,NaN,NaN,NaN,NaN,second


In [27]:
# https://stackoverflow.com/a/2415343/ with some tweaks
def weighted_avg_and_std(values, weights):
    """
    Return the weighted average and standard deviation.

    They weights are in effect first normalized so that they 
    sum to 1 (and so they must not all be 0).

    values, weights -- NumPy ndarrays with the same shape.
    """
    is_nan = np.isnan(values)
    values = values[~is_nan]
    weights = weights[~is_nan]
    average = np.average(values, weights=weights)
    # Fast and numerically precise:
    variance = np.average((values-average)**2, weights=weights)
    return pd.Series({
        'mean': average,
        'sd': np.sqrt(variance),
        # https://ngreifer.github.io/WeightIt/reference/ESS.html
        'effective_sample_size': (weights.sum() ** 2) / (weights ** 2).sum()
    })

In [28]:
# NOTE: Different from table 11.13 in report, where this is 1,542
# I tried a couple different things but couldn't figure it out
pregnant_data.anemia.notnull().sum()

np.int64(1525)

In [29]:
# Within rounding error of table 11.13 value
weighted_avg_and_std(
    pregnant_data[pregnant_data.anemia.notnull()].anemia == 'severe',
    pregnant_data[pregnant_data.anemia.notnull()].weight,
)

mean                        0.022881
sd                          0.149524
effective_sample_size    1051.325248
dtype: float64

In [30]:
# Within rounding error of table 11.13 value for any anemia
weighted_avg_and_std(
    pregnant_data[pregnant_data.anemia.notnull()].anemia.isin(['severe', 'moderate', 'mild']),
    pregnant_data[pregnant_data.anemia.notnull()].weight,
)

mean                        0.611143
sd                          0.487491
effective_sample_size    1051.325248
dtype: float64

In [31]:
assert (
    (pregnant_data[pregnant_data.anemia.notnull()].anemia == 'severe') ==
    (pregnant_data[pregnant_data.anemia.notnull()].hemoglobin_adjusted < 70)
).all()

In [32]:
assert (
    (pregnant_data[pregnant_data.anemia.notnull()].anemia.isin(['severe', 'moderate', 'mild'])) ==
    (pregnant_data[pregnant_data.anemia.notnull()].hemoglobin_adjusted < 110)
).all()

In [33]:
# Very wide age bins -- but still not enough sample size
age_bin_edges = [15, 25, 30, 50]
age_bin_edges

[15, 25, 30, 50]

In [34]:
pregnant_data["age_group"] = pd.IntervalIndex(pd.cut(pregnant_data.age, age_bin_edges, right=False))

In [35]:
# NOTE: We don't use this age stratification because it shows inconsistent patterns and fluctuations
(
    pregnant_data.groupby(["age_group", "wealth_quintile"])
        .apply(lambda df: weighted_avg_and_std(df.hemoglobin_adjusted, weights=df.weight))
        .sort_index()
)

/tmp/ipykernel_1229705/2148680057.py:3: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  pregnant_data.groupby(["age_group", "wealth_quintile"])
/tmp/ipykernel_1229705/2148680057.py:4: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda df: weighted_avg_and_std(df.hemoglobin_adjusted, weights=df.weight))


mean         sd  effective_sample_size
age_group    wealth_quintile                                              
[15.0, 25.0) lowest           103.240009  16.841299              94.676957
             second           101.444582  18.675304             114.361684
             middle           104.020304  16.915420              84.974749
             fourth           105.536957  16.225645              58.269410
             highest          108.722804  14.606053              26.898092
[25.0, 30.0) lowest           102.313692  14.476914              67.614890
             second           102.607258  14.057848              81.336717
             middle           103.686810  15.468611              65.424841
             fourth           111.145516  18.421008              36.935006
             highest          106.450793  11.268334              36.836358
[30.0, 50.0) lowest           104.612559  13.929468             114.991811
             second           103.784501  15.491811              93.520000
             middle           106.277960  14.828466              80.198700
             fourth           104.712915  14.256836              81.807489
             highest          109.394061  17.030270              76.801349

In [36]:
hemoglobin_disparities = (
    pregnant_data.groupby(["wealth_quintile"])
        .apply(lambda df: weighted_avg_and_std(df.hemoglobin_adjusted, weights=df.weight))
        .sort_index()
)
hemoglobin_disparities

/tmp/ipykernel_1229705/3994768047.py:2: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  pregnant_data.groupby(["wealth_quintile"])
/tmp/ipykernel_1229705/3994768047.py:3: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda df: weighted_avg_and_std(df.hemoglobin_adjusted, weights=df.weight))


,mean,sd,effective_sample_size
wealth_quintile,,,
lowest,103.539139,15.210946,276.303313
second,102.462408,16.636787,287.600823
middle,104.689267,15.828476,230.053871
fourth,107.325694,16.717644,152.638702
highest,108.476993,15.277522,140.496508


In [37]:
hemoglobin_disparities = hemoglobin_disparities.reset_index()
hemoglobin_disparities["sex"] = "Female"
hemoglobin_disparities = hemoglobin_disparities.set_index(["sex", "wealth_quintile"])
hemoglobin_disparities

mean         sd  effective_sample_size
sex    wealth_quintile                                              
Female lowest           103.539139  15.210946             276.303313
       second           102.462408  16.636787             287.600823
       middle           104.689267  15.828476             230.053871
       fourth           107.325694  16.717644             152.638702
       highest          108.476993  15.277522             140.496508

In [38]:
pregnancy_sim_input_data_dir = '../../0200_pregnancy_sim/src/vivarium_gates_lsff_by_wealth_quintile/data/raw_data'

In [39]:
hemoglobin_disparities["mean"].rename("value").to_csv(f'{pregnancy_sim_input_data_dir}/mean_hemoglobin_disparities/nigeria.csv')

In [40]:
hemoglobin_disparities["sd"].rename("value").to_csv(f'{pregnancy_sim_input_data_dir}/sd_hemoglobin_disparities/nigeria.csv')

### Wealth quintile probabilities

Intuitively, you might think these would be equal; but we are looking at a subpopulation (pregnancies) that skews poorer.

In [41]:
assert pregnant_data.wealth_quintile.notnull().all()

In [42]:
wealth_quintile_probabilities = []

for quintile in pregnant_data.wealth_quintile.unique():
    quintile_info = pd.DataFrame(weighted_avg_and_std(pregnant_data.wealth_quintile == quintile, pregnant_data.weight)).T
    quintile_info.insert(0, "wealth_quintile", quintile)
    wealth_quintile_probabilities.append(quintile_info)

wealth_quintile_probabilities = pd.concat(wealth_quintile_probabilities, ignore_index=True)
wealth_quintile_probabilities.sort_values('mean')

,wealth_quintile,mean,sd,effective_sample_size
0,highest,0.147899,0.355000,3048.391381
1,fourth,0.167500,0.373421,3048.391381
2,middle,0.205447,0.404028,3048.391381
4,lowest,0.235877,0.424545,3048.391381
3,second,0.243277,0.429061,3048.391381


In [43]:
wealth_quintile_probabilities['mean'].sum()

np.float64(1.0)

In [44]:
wealth_quintile_probabilities = wealth_quintile_probabilities.set_index("wealth_quintile")["mean"].to_frame().T.reset_index(drop=True)
wealth_quintile_probabilities.columns.name = None
wealth_quintile_probabilities

,highest,fourth,middle,second,lowest
0,0.147899,0.1675,0.205447,0.243277,0.235877


In [45]:
wealth_quintile_probabilities.insert(0, "sex", "Female")
wealth_quintile_probabilities

,sex,highest,fourth,middle,second,lowest
0,Female,0.147899,0.1675,0.205447,0.243277,0.235877


In [46]:
wealth_quintile_probabilities.to_csv(f'{pregnancy_sim_input_data_dir}/wealth_quintile_probabilities/nigeria.csv', index=False)

### Maternal mortality ratio

#### Maternal mortality rate

In [47]:
sibling_data.survival_status.value_counts()

survival_status
alive         193315
dead           26226
don't know        20
Name: count, dtype: int64

In [48]:
sibling_data.pregnancy_category.value_counts()

pregnancy_category
death not related          2974
died during delivery        544
died while pregnant         425
6 weeks after delivery      204
2 months after delivery      43
Name: count, dtype: int64

In [49]:
sibling_data.death_violence_or_accident.value_counts()

death_violence_or_accident
no          7147
accident     333
violence     154
Name: count, dtype: int64

In [50]:
# https://dhsprogram.com/Data/Guide-to-DHS-Statistics/Adult_Mortality_Rates.htm#Calculation1
sibling_data["exposure_start"] = np.maximum(sibling_data.date_of_birth + 12 * 15, sibling_data.interview_date - 84) # aka lowlim
# aka upplim
sibling_data["exposure_end"] = np.minimum(np.where(
    sibling_data.survival_status == 'alive',
    sibling_data.interview_date - 1,
    sibling_data.date_of_death,
), sibling_data.date_of_birth + 12 * 50 - 1)
sibling_data["exposure"] = ((sibling_data.exposure_end - sibling_data.exposure_start) + 1).clip(lower=0)

In [51]:
sibling_data.exposure.value_counts()

exposure
84.0    119939
0.0      56888
66.0      7776
42.0      6846
78.0      5633
         ...  
60.0         3
12.0         1
72.0         1
24.0         1
48.0         1
Name: count, Length: 84, dtype: int64

In [52]:
sibling_data["adult_death"] = (
    (sibling_data.survival_status == 'dead') &
    (sibling_data.date_of_death - sibling_data.date_of_birth >= 15.0 * 12) &
    (sibling_data.date_of_death - sibling_data.date_of_birth < 50.0 * 12) &
    (sibling_data.exposure > 0) &
    (sibling_data.date_of_death >= sibling_data.exposure_start) &
    (sibling_data.date_of_death <= sibling_data.exposure_end)
)

In [53]:
# Matches table 14.2
(sibling_data[sibling_data.date_of_birth.notnull()].assign(weighted_exposure=lambda df: df.exposure * df.weight).groupby("sex").weighted_exposure.sum() / 12)

sex
female    480382.344895
male      509840.974981
Name: weighted_exposure, dtype: float64

In [54]:
# Matches table 14.2
sibling_data.assign(weighted_adult_dealth=lambda df: df.adult_death * df.weight).groupby("sex").weighted_adult_dealth.sum()

sex
female    1442.193558
male      1541.674009
Name: weighted_adult_dealth, dtype: float64

In [55]:
sibling_data.death_violence_or_accident.value_counts()

death_violence_or_accident
no          7147
accident     333
violence     154
Name: count, dtype: int64

In [56]:
sibling_data.pregnancy_category.value_counts()

pregnancy_category
death not related          2974
died during delivery        544
died while pregnant         425
6 weeks after delivery      204
2 months after delivery      43
Name: count, dtype: int64

In [57]:
female_siblings = sibling_data[sibling_data.sex == 'female'].copy()
female_siblings["maternal_death"] = (
    (female_siblings.adult_death) &
    (female_siblings.pregnancy_category.isin(['died during delivery', 'died while pregnant', '6 weeks after delivery'])) &
    (~female_siblings.death_violence_or_accident.isin(['violence', 'accident']))
)

In [58]:
# Table 14.4 reports 451 maternal deaths
(female_siblings.maternal_death * female_siblings.weight).sum()

np.float64(450.75368699999996)

In [59]:
# Table 14.4 reports 480,382
(female_siblings.exposure * female_siblings.weight / 12).sum()

np.float64(480382.3448947499)

In [60]:
def maternal_mortality_rate(df):
    return ((df.maternal_death * df.weight).sum() * 1_000) / ((df.exposure * df.weight) / 12).sum()

In [61]:
# "the maternal mortality rate among women age 15-49 is 0.92 deaths per 1,000 woman-years of exposure." (p. 374)
# TODO: These are not age-standardized! We figure the *disparity* probably isn't way off.
# Should standardize according to the approach from https://github.com/LateraOlana/Fertility_SIM_DHS/blob/main/fertility/Latera_Zebb_Coworking.ipynb
maternal_mortality_rate(female_siblings)

np.float64(0.9383227585076186)

In [62]:
maternal_mortality_rates = female_siblings.groupby("wealth_quintile").apply(maternal_mortality_rate)
maternal_mortality_rates

/tmp/ipykernel_1229705/1506059063.py:1: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  maternal_mortality_rates = female_siblings.groupby("wealth_quintile").apply(maternal_mortality_rate)
/tmp/ipykernel_1229705/1506059063.py:1: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  maternal_mortality_rates = female_siblings.groupby("wealth_quintile").apply(maternal_mortality_rate)


wealth_quintile
lowest     1.460666
second     1.268344
middle     0.856495
fourth     0.787394
highest    0.485920
dtype: float64

#### General fertility rate

In [63]:
fertility_event_data = birth_data.copy()
fertility_event_data["birth_in_period"] = (
    ((fertility_event_data.interview_date - fertility_event_data.birth_date) >= 1) &
    ((fertility_event_data.interview_date - fertility_event_data.birth_date) <= 36)
)
fertility_event_data["weighted_birth_in_period"] = fertility_event_data.birth_in_period * fertility_event_data.weight

In [64]:
fertility_event_data.weighted_birth_in_period.sum()

np.float64(20018.820107)

In [65]:
fertility_event_data.groupby("wealth_quintile").weighted_birth_in_period.sum()

/tmp/ipykernel_1229705/2169524943.py:1: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  fertility_event_data.groupby("wealth_quintile").weighted_birth_in_period.sum()


wealth_quintile
lowest     4326.021447
second     4542.606853
middle     4122.133768
fourth     3727.551863
highest    3300.506176
Name: weighted_birth_in_period, dtype: float64

In [66]:
fertility_exposure_data = wra_data.copy()
fertility_exposure_data["exposure_start"] = np.maximum(fertility_exposure_data.date_of_birth + 12 * 15, fertility_exposure_data.interview_date - 36) # aka lowlim
# aka upplim
fertility_exposure_data["exposure_end"] = np.minimum(fertility_exposure_data.interview_date - 1, fertility_exposure_data.date_of_birth + 12 * 45)
fertility_exposure_data["exposure"] = ((fertility_exposure_data.exposure_end - fertility_exposure_data.exposure_start) + 1).clip(lower=0)
fertility_exposure_data["weighted_exposure"] = fertility_exposure_data.exposure * fertility_exposure_data.weight

In [67]:
# Within rounding error of value reported in Table 5.1
fertility_event_data.weighted_birth_in_period.sum() * 1_000 / (fertility_exposure_data.weighted_exposure.sum() / 12)

np.float64(181.80045597312596)

In [68]:
gfr_by_wealth = (
    fertility_event_data.groupby("wealth_quintile").weighted_birth_in_period.sum() * 1_000 /
    (fertility_exposure_data.groupby("wealth_quintile").weighted_exposure.sum() / 12)
)
gfr_by_wealth

/tmp/ipykernel_1229705/4161804572.py:2: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  fertility_event_data.groupby("wealth_quintile").weighted_birth_in_period.sum() * 1_000 /
/tmp/ipykernel_1229705/4161804572.py:3: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  (fertility_exposure_data.groupby("wealth_quintile").weighted_exposure.sum() / 12)


wealth_quintile
lowest     228.551509
second     214.895278
middle     191.264704
fourth     158.111396
highest    132.443590
dtype: float64

In [69]:
maternal_disorders_incidence_disparities = (maternal_mortality_rates / gfr_by_wealth).rename("value").rename_axis("wealth_quintile").reset_index()
maternal_disorders_incidence_disparities.insert(0, "sex", "Female")
maternal_disorders_incidence_disparities

,sex,wealth_quintile,value
0,Female,lowest,0.006391
1,Female,second,0.005902
2,Female,middle,0.004478
3,Female,fourth,0.004980
4,Female,highest,0.003669


In [70]:
maternal_disorders_incidence_disparities.to_csv(f'{pregnancy_sim_input_data_dir}/maternal_disorders_incidence_disparities/nigeria.csv', index=False)